In [ ]:
from pyspark.sql.functions import (
    col, to_timestamp, unix_timestamp, when, lit, round, 
    sum as spark_sum, abs, isnan, count, sha2, concat_ws, hour, dayofweek
)
from pyspark.sql.types import IntegerType, DoubleType, StringType,TimestampType

# ============================================================
# STEP 1: Read Bronze
# ============================================================
bronze_df = spark.table("nyc_taxi.bronze.green_taxi")

# ============================================================
# STEP 2: Cast and Standardize Data Types
# ============================================================


cast_rules = {
    # Timestamps
    "lpep_pickup_datetime": TimestampType(),
    "lpep_dropoff_datetime": TimestampType(),
    # Numeric IDs
    "VendorID": IntegerType(), 
    "RatecodeID": IntegerType(),
    "payment_type": IntegerType(),  
    "passenger_count": IntegerType(),   
    # Monetary fields
    "fare_amount": DoubleType(),
    "extra": DoubleType(),   
    "mta_tax": DoubleType(),  
    "improvement_surcharge": DoubleType(),    
    "tip_amount": DoubleType(),    
    "tolls_amount": DoubleType(),    
    "total_amount": DoubleType(),    
    "trip_distance": DoubleType(),    
    # Coordinates
    "pickup_longitude": DoubleType(),
    "pickup_latitude": DoubleType(),    
    "dropoff_longitude": DoubleType(),    
    "dropoff_latitude": DoubleType(),    

}


df = bronze_df
for column_name, data_type in cast_rules.items():
    if column_name in df.columns:
        df = df.withColumn(column_name, col(column_name).cast(data_type))
    else:
        # Optional: add a default null column so downstream code doesn't break
        df = df.withColumn(column_name, lit(None).cast(data_type))

silver_df = df

# ============================================================
# STEP 3: Clean Nulls and Invalid Values
# ============================================================

# silver_df_cached_std = silver_df.cache()


silver_df = (silver_df
    # Passenger count: null or 0 → assume 1; cap at 6
    .withColumn("passenger_count", 
        when(col("passenger_count").isNull() | (col("passenger_count") <= 0), lit(1))
        .when(col("passenger_count") > 6, lit(6))
        .otherwise(col("passenger_count"))
    )
    
    # Payment type: null or outside 1-6 → 5 (Unknown)
    .withColumn("payment_type",
        when(col("payment_type").isNull() | (col("payment_type") < 1) | (col("payment_type") > 6), lit(5))
        .otherwise(col("payment_type"))
    )
    
    # Rate code: null or outside 1-6 → 1 (Standard)
    .withColumn("RatecodeID",
        when(col("RatecodeID").isNull() | (col("RatecodeID") < 1) | (col("RatecodeID") > 6), lit(1))
        .otherwise(col("RatecodeID"))
    )
)

# ============================================================
# STEP 4: Derive New Columns (Business Logic)
# ============================================================
# silver_df_cached_clean = silver_df.cache()

silver_df = (silver_df
    .withColumn('trip_id', 
        sha2(concat_ws("||", col("lpep_pickup_datetime"), col("lpep_dropoff_datetime"), col("PULocationID"),col("DOLocationID") ),256)
    )
    # Trip duration in minutes
    .withColumn("trip_duration_minutes",
        round((unix_timestamp(col("lpep_dropoff_datetime")) - unix_timestamp(col("lpep_pickup_datetime"))) / 60, 2)
    )
    
    # Fare per mile (for anomaly detection)
    .withColumn("fare_per_mile",
        when(col("trip_distance") > 0, round(col("fare_amount") / col("trip_distance"), 2))
        .otherwise(lit(None))
    )
    
    # Tip percentage (credit card tips only; cash is 0 in raw data)
    .withColumn("tip_percentage",
        when(col("fare_amount") > 0, round((col("tip_amount") / col("fare_amount")) * 100, 2))
        .otherwise(lit(0.0))
    )
    
    # Hour of day and day of week for analytics
    .withColumn("pickup_hour", hour(col("lpep_pickup_datetime")))
    .withColumn("pickup_day_of_week", dayofweek(col("lpep_pickup_datetime")))
)

# ============================================================
# STEP 5: Filter Invalid Rows (Data Quality)
# ============================================================


silver_df = (silver_df
    # Drop rows with impossible timestamps
    .filter(col("lpep_pickup_datetime").isNotNull())
    .filter(col("lpep_dropoff_datetime").isNotNull())
    .filter(col("lpep_dropoff_datetime") > col("lpep_pickup_datetime"))
    
    # Drop rows with null island coordinates or out-of-range NYC
    #.filter((col("pickup_latitude") != 0) & (col("pickup_longitude") != 0))
    #.filter((col("pickup_latitude").between(40.4, 41.0)))
    #.filter((col("pickup_longitude").between(-74.3, -73.6)))
    
    # Drop rows with zero or negative distance
    #.filter(col("trip_distance") > 0)
    
    # Drop rows with negative monetary amounts (data corruption)
    .filter(col("fare_amount") >= 0)
    .filter(col("total_amount") >= 0)
    
    # Drop rows where trip duration is negative or > 24 hours (likely bad data)
    .filter((col("trip_duration_minutes") > 0) & (col("trip_duration_minutes") <= 1440))
)



# ============================================================
# STEP 7: Select Final Columns (Drop Raw String Columns)
# ============================================================

silver_df = silver_df.select(
    "trip_id",
    col("VendorID").alias("vendor_id"),
    col("lpep_pickup_datetime").alias("pickup_datetime"),
    col("lpep_dropoff_datetime").alias("dropoff_datetime"),
    "trip_duration_minutes",
    "passenger_count",
    "trip_distance",
    "fare_per_mile",
    "PULocationID",
    "DOLocationID",
    col("RatecodeID").alias("rate_code_id"),
    "store_and_fwd_flag",
    "payment_type",
    "tip_amount",
    "tip_percentage",
    "fare_amount",
    "extra",
    "mta_tax",
    "improvement_surcharge",
    "tolls_amount",
    "total_amount",
    "trip_type",
    "pickup_hour",
    "pickup_day_of_week"
)

# ============================================================
# STEP 8: Write to Silver Table
# ============================================================


from delta.tables import DeltaTable

table_name = "nyc_taxi.silver.green_taxi"

# Check if the table exists in the catalog
if spark.catalog.tableExists(table_name):
    # Table exists: Load by name and upsert
    target_table = DeltaTable.forName(spark, table_name)
    
    target_table.alias("target").merge(
        silver_df.alias("source"), 
        "target.trip_id = source.trip_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    
else:
    # Table does not exist: Save the dataframe to create it
    silver_df.write.format("delta").mode("saveAsTable").saveAsTable(table_name)
